# 04. PER Relative Valuation

Derive Samyang Foods' PER premium from observable peer multiples rather than assigning an arbitrary premium.

**Primary peers:** Nongshim and Orion. Secondary peers are displayed only as a reasonableness check.

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path('../data/processed')
peers = pd.read_csv(DATA / 'peer_valuation_2026_2027.csv')
peers

In [ ]:
primary = peers[peers['role'] == 'primary peer'].copy()
samyang = peers[peers['role'] == 'subject'].iloc[0]

peer_pe_2026 = primary['pe_2026'].median()
peer_pe_2027 = primary['pe_2027'].median()
premium_2026 = samyang['pe_2026'] / peer_pe_2026 - 1
premium_2027 = samyang['pe_2027'] / peer_pe_2027 - 1
normalized_premium = (premium_2026 + premium_2027) / 2

print(f'Primary-peer median P/E 2026E: {peer_pe_2026:.2f}x')
print(f'Observed Samyang premium 2026E: {premium_2026:.1%}')
print(f'Primary-peer median P/E 2027E: {peer_pe_2027:.3f}x')
print(f'Observed Samyang premium 2027E: {premium_2027:.1%}')
print(f'Normalized observed premium: {normalized_premium:.1%}')

## Target-multiple sensitivity

The Base case uses the normalized observed premium (~50%). Bear/Bull are sensitivity bands ±15 percentage points around Base.

In [ ]:
project_eps_2027 = 92_839
consensus_eps_2027 = 91_345

premium_cases = {'Bear': 0.35, 'Base': 0.50, 'Bull': 0.65}
rows = []
for scenario, premium in premium_cases.items():
    target_pe = peer_pe_2027 * (1 + premium)
    rows.append({
        'scenario': scenario,
        'premium': premium,
        'target_pe': target_pe,
        'implied_price_project_eps': target_pe * project_eps_2027,
        'implied_price_consensus_eps': target_pe * consensus_eps_2027,
    })

valuation = pd.DataFrame(rows)
valuation

## Interpretation rule

- Do not treat the PER result as an independent truth if the premium itself is anchored to current market pricing.
- Use PER as a **relative-value cross-check** against DCF and RIM.
- If the final DCF/RIM outputs diverge materially, explain whether the gap comes from growth, margins, WACC, terminal assumptions, or the peer premium.